# VisDrone prompt app: type a question, get boxes and an answer

**How it works**
1. **Detector** (once per image, cached): fixed SAHI from evaluation v2 = 512 px slices at 2x zoom + whole image at 1536,
   NMS 0.6, boundary filter. Two modes: **Fast** = K1, **Accurate** = K1 + E2 fused with WBF 0.65.
2. **Parser**: Qwen2.5-1.5B-Instruct runs locally (no API key) and turns the text into
   `{task, targets, region}`. A rule-based parser is the fallback and can be chosen in the app for comparison.
3. **Word mapping**: user words → VisDrone classes ("bike" → bicycle, "people" → pedestrian + people, "vehicles" → 7 classes).
4. **Executor**: filters the cached boxes by class, region and confidence, and writes the answer.
5. **Unknown words** (not one of the 10 classes) → zero-shot YOLOE-26m, **marked experimental**.

**Run:** GPU T4×2, Internet ON, Run All. Attach the **visdrone-training** output (K1) and the **E2 notebook** output.
The last cell prints a public `gradio.live` link (valid while the notebook runs) and also shows the app inline.
Precomputing val for both detector modes takes ~15 min the first time; it is optional (images are otherwise detected on first use).

In [1]:
# --- Setup ---------------------------------------------------------------
!pip install -q "ultralytics==8.4.174" "sahi==0.12.8" ensemble-boxes "gradio==6.30.0" "transformers>=4.45" accelerate "git+https://github.com/ultralytics/CLIP.git"
import os, io, re, json, time, pickle, logging, hashlib, zipfile, urllib.request
from pathlib import Path
import numpy as np, torch, cv2
from ultralytics import YOLO, YOLOE
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)

ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
DATA = Path("/tmp/VisDrone") if Path("/kaggle").exists() else ROOT / "VisDrone"
APP = ROOT / "app"; CACHE = APP / "cache"
for d in (DATA, APP, CACHE): d.mkdir(parents=True, exist_ok=True)
NGPU = torch.cuda.device_count()
DET_DEV = 0 if NGPU else "cpu"                                 # detectors on GPU 0
LLM_DEV = "cuda:1" if NGPU > 1 else ("cuda:0" if NGPU else "cpu")   # Qwen on GPU 1 when there are two
HALF = NGPU > 0
print("GPUs:", NGPU, "| detectors on", DET_DEV, "| LLM on", LLM_DEV)

  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/46.9 kB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 591.1 kB/s eta 0:00:00


   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/1.4 MB 4.3 MB/s eta 0:00:01

... (64 outputs left out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


GPUs: 2 | detectors on 0 | LLM on cuda:1


In [2]:
# --- Config -----------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")
def find(pattern):
    hits = sorted(INPUT_ROOT.rglob(pattern))
    return hits[0] if hits else None

WEIGHTS = {"K1": find("K1_y26m_tiles/weights/best.pt"), "E2": find("yoloe26m_e2_full.pt")}
YOLOE_ZS = "yoloe-26m-seg.pt"                  # zero-shot open-vocab model (auto-downloads)
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"        # local, no API key; 3B also fits a T4 if you want better parsing
for k, v in WEIGHTS.items(): print(k, v)
assert WEIGHTS["K1"], "Attach the visdrone-training output (K1)"

NAMES = ["pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle", "awning-tricycle", "bus", "motor"]
CONF_FLOOR = 0.01            # detector keeps everything above this; the app's slider decides what is shown/counted
OVERLAP = 0.25
SLICE_BATCH = 16
# Display/count thresholds per mode. Placeholders until the calibration cell below runs (it overwrites them).
COUNT_THR = {"fast": 0.35, "accurate": 0.30}
MODES = {"fast": "Fast: K1 + SAHI (~0.3 s)", "accurate": "Accurate: K1 + E2 ensemble (~1 s)"}

K1 /kaggle/input/notebooks/ciiphersiren/visdrone-training/runs/K1_y26m_tiles/weights/best.pt
E2 /kaggle/input/notebooks/ciiphersiren/yoloe-2/yoloe26m_e2_full.pt


In [3]:
# --- VisDrone val + test-dev (images + labels, for picking images and showing ground truth) ---
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
SPLIT_DIRS = {"val": DATA / "VisDrone2019-DET-val", "test-dev": DATA / "VisDrone2019-DET-test-dev"}
for split, d in SPLIT_DIRS.items():
    if not d.exists():
        z = DATA / f"{d.name}.zip"; print("Downloading", d.name)
        urllib.request.urlretrieve(BASE + z.name, z)
        with zipfile.ZipFile(z) as f: f.extractall(DATA)
        z.unlink()

def load_gt(split):
    # per image: list of (cls0..9, x1, y1, x2, y2); ignored regions and 'others' dropped
    out = {}
    for ann in sorted((SPLIT_DIRS[split] / "annotations").glob("*.txt")):
        rows = []
        for line in ann.read_text().strip().splitlines():
            v = line.strip().rstrip(",").split(",")
            if len(v) < 6: continue
            x, y, w, h, score, cat = map(int, v[:6])
            if w <= 0 or h <= 0 or score == 0 or cat in (0, 11): continue
            rows.append((cat - 1, x, y, x + w, y + h))
        out[f"{ann.stem}.jpg"] = rows
    return out

GT = {s: load_gt(s) for s in SPLIT_DIRS}
# Image picker choices: "val/0000001_02999_d_0000005.jpg (123 objects)", densest first
CHOICES = []
for s in ("val", "test-dev"):
    for f, rows in sorted(GT[s].items(), key=lambda kv: -len(kv[1])):
        CHOICES.append(f"{s}/{f} ({len(rows)} objects)")
print(len(CHOICES), "VisDrone images available in the picker")

2158 VisDrone images available in the picker


In [4]:
# --- Detector engine (same settings as evaluation v2's best configs) --------
from sahi.slicing import get_slice_bboxes
from sahi.postprocess.combine import batched_nms
from ensemble_boxes import weighted_boxes_fusion

DET = {}
def detector(tag):
    if tag not in DET:
        DET[tag] = YOLO(str(WEIGHTS[tag]))
        DET[tag].predict(np.zeros((640, 640, 3), np.uint8), imgsz=640, half=HALF, device=DET_DEV, verbose=False)
    return DET[tag]

def _arr(r):
    b = r.boxes
    if b is None or len(b) == 0: return np.zeros((0, 6), np.float32)
    return np.c_[b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()].astype(np.float32)

def _edge_filter(d, sb, W, H, margin=2.0):
    # drop slice boxes touching an interior slice edge (cut objects); see evaluation v2
    if len(d) == 0: return d
    x1, y1, x2, y2 = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    sx1, sy1, sx2, sy2 = sb[:, 0], sb[:, 1], sb[:, 2], sb[:, 3]
    cut = (((sx1 > 0) & (x1 <= sx1 + margin)) | ((sy1 > 0) & (y1 <= sy1 + margin)) |
           ((sx2 < W) & (x2 >= sx2 - margin)) | ((sy2 < H) & (y2 >= sy2 - margin)))
    return d[~cut]

def sahi_detect(model, img, slice_size=512, slice_imgsz=1024, full_imgsz=1536, nms_thr=0.6):
    H, W = img.shape[:2]
    full = _arr(model.predict(img, imgsz=full_imgsz, conf=CONF_FLOOR, max_det=1000, half=HALF, device=DET_DEV, verbose=False)[0])
    boxes = get_slice_bboxes(image_height=H, image_width=W, slice_height=slice_size, slice_width=slice_size,
                             auto_slice_resolution=False, overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP)
    crops = [np.ascontiguousarray(img[y1:y2, x1:x2]) for x1, y1, x2, y2 in boxes]
    parts = [full]
    for i in range(0, len(crops), SLICE_BATCH):
        rs = model.predict(crops[i:i + SLICE_BATCH], imgsz=slice_imgsz, conf=CONF_FLOOR, max_det=1000, half=HALF, device=DET_DEV, verbose=False)
        for r, (x1, y1, x2, y2) in zip(rs, boxes[i:i + SLICE_BATCH]):
            a = _arr(r); a[:, [0, 2]] += x1; a[:, [1, 3]] += y1
            parts.append(_edge_filter(a, np.tile(np.float32([x1, y1, x2, y2]), (len(a), 1)), W, H))
    P = np.concatenate(parts)
    return P[batched_nms(P, match_metric="IOU", match_threshold=nms_thr)] if len(P) else P

def wbf(dets, W, H, thr=0.65):
    bl = [np.clip(d[:, :4] / [W, H, W, H], 0, 1).tolist() for d in dets]
    b, s, l = weighted_boxes_fusion(bl, [d[:, 4].tolist() for d in dets], [d[:, 5].tolist() for d in dets],
                                    iou_thr=thr, skip_box_thr=CONF_FLOOR)
    return np.c_[np.asarray(b) * [W, H, W, H], s, l].astype(np.float32).reshape(-1, 6)

def detect(img, mode):
    # img: BGR uint8. Returns (N,6) [x1,y1,x2,y2,score,cls0..9] and seconds.
    t0 = time.perf_counter()
    if mode == "fast" or WEIGHTS["E2"] is None:
        d = sahi_detect(detector("K1"), img)
    else:
        H, W = img.shape[:2]
        d = wbf([sahi_detect(detector("K1"), img), sahi_detect(detector("E2"), img)], W, H)
    if NGPU: torch.cuda.synchronize()
    return d, time.perf_counter() - t0

# ---- Saved detections: one pickle per (image, mode); uploads are keyed by a hash of their pixels ----
def image_key(img=None, choice=None):
    if choice: return choice.split(" (")[0].replace("/", "__")
    return "upload__" + hashlib.md5(img.tobytes()).hexdigest()[:16]

def cached_detect(img, mode, key):
    f = CACHE / f"{key}__{mode}.pkl"
    if f.exists():
        d, t = pickle.load(open(f, "rb")); return d, t, True
    d, t = detect(img, mode); pickle.dump((d, t), open(f, "wb"))
    return d, t, False

def load_choice(choice):
    split, rest = choice.split("/", 1); fname = rest.split(" (")[0]
    return split, fname, cv2.imread(str(SPLIT_DIRS[split] / "images" / fname))

In [5]:
# --- Optional: precompute val for both modes (makes the picker instant; also feeds the calibration) ---
PRECOMPUTE_VAL = True
if PRECOMPUTE_VAL:
    from tqdm.auto import tqdm
    val_choices = [c for c in CHOICES if c.startswith("val/")]
    for mode in (["fast", "accurate"] if WEIGHTS["E2"] else ["fast"]):
        ts = []
        for c in tqdm(val_choices, desc=f"val {mode}"):
            _, _, img = load_choice(c)
            _, t, _ = cached_detect(img, mode, image_key(choice=c)); ts.append(t)
        print(f"{mode}: {1000*np.median(ts):.0f} ms/img median (cached runs count as their original time)")

val fast:   0%|          | 0/548 [00:00<?, ?it/s]

In [ ]:
# --- Calibrate the counting threshold on val (per mode) --------------------
# AP uses every box down to conf 0.01; counting needs a cut-off. Pick the one with the lowest mean absolute
# count error per (image, class) on val. This is the first number for the prompting evaluation.
def count_error(mode, thr, split="val"):
    errs, rel = [], []
    for c in [c for c in CHOICES if c.startswith(split + "/")]:
        f = CACHE / f"{image_key(choice=c)}__{mode}.pkl"
        if not f.exists(): continue
        d, _ = pickle.load(open(f, "rb")); fname = c.split("/", 1)[1].split(" (")[0]
        gt = np.bincount([r[0] for r in GT[split][fname]], minlength=10)
        pr = np.bincount(d[d[:, 4] >= thr, 5].astype(int), minlength=10)[:10]
        errs += list(np.abs(pr - gt)); rel += [abs(p - g) / g for p, g in zip(pr, gt) if g > 0]
    return (np.mean(errs) if errs else np.nan), (np.mean(rel) if rel else np.nan)

import pandas as pd
rows = []
for mode in (["fast", "accurate"] if WEIGHTS["E2"] else ["fast"]):
    for thr in np.round(np.arange(0.05, 0.71, 0.05), 2):
        mae, mre = count_error(mode, thr)
        rows.append(dict(mode=mode, thr=thr, count_MAE=round(mae, 3), rel_err=round(mre, 3)))
cal = pd.DataFrame(rows).dropna()
if len(cal):
    for mode, g in cal.groupby("mode"):
        best = g.sort_values("count_MAE").iloc[0]
        COUNT_THR[mode] = float(best.thr)
        print(f"{mode}: best counting threshold {best.thr:.2f} -> mean abs count error {best.count_MAE:.2f} per image-class, "
              f"mean relative error {best.rel_err:.1%}")
    cal.to_csv(APP / "count_threshold_calibration.csv", index=False)
print("COUNT_THR =", COUNT_THR)
cal.pivot(index="thr", columns="mode", values="count_MAE") if len(cal) else None

In [ ]:
# --- Word mapping: user words -> VisDrone classes ---------------------------
SYN = {
    "pedestrian": ["pedestrian", "pedestrians", "walker", "walkers", "person walking", "people walking", "walking"],
    "people": ["sitting", "seated", "person sitting", "people sitting", "sitting people"],
    "bicycle": ["bicycle", "bicycles", "bike", "bikes", "cycle", "cycles", "cyclist", "cyclists"],
    "car": ["car", "cars", "sedan", "sedans", "suv", "suvs", "taxi", "taxis", "hatchback"],
    "van": ["van", "vans", "minivan", "minivans", "minibus"],
    "truck": ["truck", "trucks", "lorry", "lorries", "pickup", "pickups"],
    "tricycle": ["tricycle", "tricycles", "trike", "trikes", "cargo tricycle"],
    "awning-tricycle": ["awning-tricycle", "awning tricycle", "awning-tricycles", "awning tricycles", "auto rickshaw",
                        "auto rickshaws", "autorickshaw", "rickshaw", "rickshaws", "tuk tuk", "tuk-tuk", "tricycle with a canopy"],
    "bus": ["bus", "buses", "busses", "coach", "coaches"],
    "motor": ["motor", "motors", "motorcycle", "motorcycles", "motorbike", "motorbikes", "scooter", "scooters", "moped", "mopeds"],
}
GROUPS = {   # words that mean several classes
    "people": ["pedestrian", "people"], "person": ["pedestrian", "people"], "persons": ["pedestrian", "people"],
    "humans": ["pedestrian", "people"], "human": ["pedestrian", "people"], "crowd": ["pedestrian", "people"],
    "vehicles": ["car", "van", "truck", "bus", "tricycle", "awning-tricycle", "motor"],
    "vehicle": ["car", "van", "truck", "bus", "tricycle", "awning-tricycle", "motor"],
    "two-wheelers": ["bicycle", "motor"], "two wheelers": ["bicycle", "motor"],
    "everything": NAMES, "all objects": NAMES, "objects": NAMES, "anything": NAMES,
}
WORD2CLS = {w: [c] for c, ws in SYN.items() for w in ws}
WORD2CLS.update(GROUPS)

def map_word(w):
    # -> (list of VisDrone classes, or None if it is an open-vocab word)
    w = w.lower().strip().strip(".?!,")
    if w in WORD2CLS: return WORD2CLS[w]
    if w.endswith("s") and w[:-1] in WORD2CLS: return WORD2CLS[w[:-1]]
    return None

In [ ]:
# --- Parsers: Qwen (local) and a rule-based fallback ------------------------
TASKS = ["detect", "count", "exists", "compare"]
REGIONS = ["all", "left", "right", "top", "bottom", "center", "top-left", "top-right", "bottom-left", "bottom-right"]

def rule_parse(text):
    t = " " + text.lower() + " "
    task = ("compare" if re.search(r"\bmore\b|\bfewer\b|\bless\b|\bcompare\b|\bvs\b|\bversus\b", t) else
            "count" if re.search(r"how many|\bcount\b|number of", t) else
            "exists" if re.search(r"\bis there\b|\bare there\b|\bany\b|\bdo you see\b", t) else "detect")
    region = "all"
    for r in sorted(REGIONS[1:], key=len, reverse=True):
        if re.search(rf"\b{r.replace('-', '[ -]')}\b", t): region = r; break
    targets = []
    for w in sorted(WORD2CLS, key=len, reverse=True):          # longest known phrases first
        if re.search(rf"\b{re.escape(w)}\b", t) and not any(w in x for x in targets):
            targets.append(w)
    if not targets:   # unknown object: take the words after the verb, drop filler
        m = re.search(r"(?:show|find|detect|count|locate|highlight|where are|where is|how many|is there an?|are there|any)\s+(?:me\s+)?(?:all\s+)?(?:the\s+)?(.+)", t)
        if m:
            obj = re.sub(r"\b(in|on|at|of)\b.*$", "", m.group(1))
            obj = re.sub(r"\b(are|is|there|please|image|picture|photo|here)\b", "", obj).strip(" ?.!")
            if obj: targets = [obj]
    return {"task": task, "targets": targets or ["everything"], "region": region}

LLM = {"tok": None, "model": None, "error": None}
def load_llm():
    if LLM["model"] is None and LLM["error"] is None:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            LLM["tok"] = AutoTokenizer.from_pretrained(LLM_NAME)
            LLM["model"] = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16 if NGPU else torch.float32).to(LLM_DEV).eval()
        except Exception as e:
            LLM["error"] = repr(e); print("Qwen not available, using the rule parser:", e)
    return LLM["model"] is not None

SYSTEM = (
    "You turn a user's request about a drone image into JSON. Reply with JSON only, no text.\n"
    'Schema: {"task": one of ["detect","count","exists","compare"], "targets": [object words as the user said them], '
    '"region": one of ["all","left","right","top","bottom","center","top-left","top-right","bottom-left","bottom-right"]}\n'
    "detect = show/find/highlight; count = how many; exists = is there/are there; compare = which is more.\n"
    "Keep each target as a short noun phrase in the user's words (singular or plural is fine). Use region \"all\" unless a part of the image is named."
)
FEWSHOT = [
    ("how many cars are on the left?", {"task": "count", "targets": ["cars"], "region": "left"}),
    ("show me all the buses and trucks", {"task": "detect", "targets": ["buses", "trucks"], "region": "all"}),
    ("are there more motorbikes or bicycles?", {"task": "compare", "targets": ["motorbikes", "bicycles"], "region": "all"}),
    ("is there a traffic cone near the top right", {"task": "exists", "targets": ["traffic cone"], "region": "top-right"}),
]

def llm_parse(text):
    tok, model = LLM["tok"], LLM["model"]
    msgs = [{"role": "system", "content": SYSTEM}]
    for q, a in FEWSHOT:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": json.dumps(a)}]
    msgs.append({"role": "user", "content": text})
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(LLM_DEV)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=96, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", raw, re.S)
    q = json.loads(m.group(0))
    q = {"task": q.get("task", "detect"), "targets": [str(x) for x in q.get("targets", [])] or ["everything"],
         "region": q.get("region", "all")}
    if q["task"] not in TASKS: q["task"] = "detect"
    if q["region"] not in REGIONS: q["region"] = "all"
    return q, raw

def parse(text, use_llm=True):
    if use_llm and load_llm():
        try:
            q, raw = llm_parse(text); return q, "Qwen", raw
        except Exception as e:
            q = rule_parse(text); return q, "rules (Qwen output unreadable)", repr(e)
    return rule_parse(text), "rules", ""

load_llm()
for s in ["how many cars on the right side?", "show me the bikes", "is there a construction crane?", "more vans or trucks?"]:
    print(s, "->", parse(s))

In [ ]:
# --- Executor: filter saved boxes, YOLOE fallback, answer text, drawing ----
ZS = {"model": None}
def yoloe_zero_shot(img, words):
    # open-vocab fallback: whole image at 1536 (SAHI hurt zero-shot YOLOE in our tests)
    if ZS["model"] is None: ZS["model"] = YOLOE(YOLOE_ZS)
    ZS["model"].set_classes(list(words))
    r = ZS["model"].predict(img, imgsz=1536, conf=0.05, max_det=300, half=HALF, device=DET_DEV, verbose=False)[0]
    return _arr(r)

def in_region(d, region, W, H):
    if region == "all" or len(d) == 0: return np.ones(len(d), bool)
    cx, cy = (d[:, 0] + d[:, 2]) / 2, (d[:, 1] + d[:, 3]) / 2
    L, R, T, B = cx < W / 2, cx >= W / 2, cy < H / 2, cy >= H / 2
    return {"left": L, "right": R, "top": T, "bottom": B,
            "center": (cx > W / 3) & (cx < 2 * W / 3) & (cy > H / 3) & (cy < 2 * H / 3),
            "top-left": T & L, "top-right": T & R, "bottom-left": B & L, "bottom-right": B & R}.get(region, np.ones(len(d), bool))

PALETTE = [(56, 56, 255), (151, 157, 255), (31, 112, 255), (29, 178, 255), (49, 210, 207),
           (10, 249, 72), (23, 204, 146), (134, 219, 61), (211, 188, 0), (255, 115, 100)]

def draw(img, groups):
    # groups: list of (label, boxes (N,6), colour). Thin boxes; text labels only when few boxes.
    out = img.copy(); n = sum(len(g[1]) for g in groups)
    for label, d, col in groups:
        for x1, y1, x2, y2, s, c in d:
            cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
            if n <= 60:
                cv2.putText(out, f"{label} {s:.2f}", (int(x1), max(10, int(y1) - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1, cv2.LINE_AA)
    return out[:, :, ::-1]   # RGB for Gradio

def answer(task, counts, region):
    where = "" if region == "all" else f" in the {region.replace('-', ' ')} part"
    names = list(counts)
    if task == "exists":
        return " ".join(f"{'Yes' if counts[n] else 'No'}: {counts[n]} {n}{where}." for n in names)
    if task == "compare" and len(names) >= 2:
        a, b = names[0], names[1]
        if counts[a] == counts[b]: return f"Same number of {a} and {b}{where}: {counts[a]} each."
        hi, lo = (a, b) if counts[a] > counts[b] else (b, a)
        return f"More {hi} ({counts[hi]}) than {lo} ({counts[lo]}){where}."
    return " ".join(f"{counts[n]} {n}{where}." for n in names)

def run_query(img, key, gt_rows, text, mode, use_llm, thr):
    q, parser, raw = parse(text, use_llm)
    H, W = img.shape[:2]
    d, det_t, cached = cached_detect(img, mode, key)
    groups, counts, notes, gt_counts = [], {}, [], {}
    for word in q["targets"]:
        classes = map_word(word)
        if classes is not None and set(classes) == set(NAMES):      # "everything": one group per class
            for k, c in enumerate(NAMES):
                sel = d[(d[:, 4] >= thr) & (d[:, 5].astype(int) == k)]
                sel = sel[in_region(sel, q["region"], W, H)]
                if len(sel): counts[c] = len(sel); groups.append((c, sel, PALETTE[k]))
                if gt_rows is not None:
                    g = np.array([r[1:] + (1.0, r[0]) for r in gt_rows if r[0] == k], np.float32).reshape(-1, 6)
                    if len(g) or len(sel): gt_counts[c] = int(in_region(g, q["region"], W, H).sum())
                    if c not in counts and gt_counts.get(c): counts[c] = 0
        elif classes is not None:
            ids = [NAMES.index(c) for c in classes]
            sel = d[(d[:, 4] >= thr) & np.isin(d[:, 5].astype(int), ids)]
            sel = sel[in_region(sel, q["region"], W, H)]
            label = word.lower().strip(" ?.!")
            counts[label] = len(sel)
            groups.append((label, sel, PALETTE[ids[0] % 10]))
            if gt_rows is not None:
                g = np.array([r[1:] + (1.0, r[0]) for r in gt_rows if r[0] in ids], np.float32).reshape(-1, 6)
                gt_counts[label] = int(in_region(g, q["region"], W, H).sum())
            if [label] != classes:
                notes.append(f"'{word}' read as VisDrone class{'es' if len(classes) > 1 else ''}: {', '.join(classes)}")
        else:
            t0 = time.perf_counter()
            z = yoloe_zero_shot(img, [word]); z = z[in_region(z, q["region"], W, H)]
            z = z[z[:, 4] >= 0.15]
            counts[word] = len(z)
            groups.append((word, z, (0, 200, 255)))
            notes.append(f"'{word}' is not a VisDrone class → zero-shot YOLOE (experimental, unreliable on aerial images), "
                         f"{1000*(time.perf_counter()-t0):.0f} ms")
    text_answer = answer(q["task"], counts, q["region"])
    return draw(img, groups), text_answer, counts, gt_counts, q, parser, raw, det_t, cached, notes

# quick test on the densest val image
c0 = CHOICES[0]; _, f0, im0 = load_choice(c0)
r = run_query(im0, image_key(choice=c0), GT["val"][f0], "how many cars are on the left?", "fast", True, COUNT_THR["fast"])
print(c0, "|", r[1], "| ground truth:", r[3], "| parser:", r[5], r[4], f"| detector {r[7]*1000:.0f} ms, cached={r[8]}")

In [ ]:
# --- The app -----------------------------------------------------------------
import gradio as gr

EXAMPLES = ["how many cars are there?", "show me the buses and trucks", "are there more motorbikes or bicycles?",
            "how many people on the left?", "is there a traffic cone?", "show all vehicles in the top right"]

def pick(choice):
    if not choice: return gr.update(), None      # cleared by an upload: keep the uploaded image
    split, fname, img = load_choice(choice)
    return img[:, :, ::-1], {"split": split, "file": fname, "choice": choice}

def on_upload(_img):
    return None, gr.update(value=None)

def handle(img_rgb, picked, text, mode_label, use_llm, thr):
    if img_rgb is None: return None, None, "Pick a VisDrone image or upload one.", None
    if not text.strip(): text = "show everything"
    img = np.ascontiguousarray(img_rgb[:, :, ::-1])
    if picked:
        key, gt_rows = image_key(choice=picked["choice"]), GT[picked["split"]][picked["file"]]
    else:
        key, gt_rows = image_key(img=img), None
    modes = ["fast", "accurate"] if mode_label == "Compare both" else [m for m, l in MODES.items() if l == mode_label]
    if "accurate" in modes and WEIGHTS["E2"] is None: modes = ["fast"]
    outs, lines = [], []
    for m in modes:
        t = COUNT_THR[m] if thr is None or thr < 0 else thr
        vis, ans, counts, gtc, q, parser, raw, det_t, cached, notes = run_query(img, key, gt_rows, text, m, use_llm, t)
        outs.append(vis)
        lines.append(f"### {MODES[m].split(' (')[0]}\n**{ans}**")
        tbl = "| Target | Detected | Ground truth |\n|---|---|---|\n" + "\n".join(
            f"| {k} | {v} | {gtc.get(k, '–')} |" for k, v in counts.items())
        lines.append(tbl)
        lines.append(f"Detector {'(saved) ' if cached else ''}{det_t*1000:.0f} ms · count threshold {t:.2f}")
        lines += [f"- {n}" for n in notes]
    diag = {"parser": parser, "parsed": q, "raw_llm_output": raw}
    second = outs[1] if len(outs) > 1 else None
    return outs[0], gr.update(value=second, visible=second is not None), "\n\n".join(lines), diag

with gr.Blocks(title="VisDrone prompt detection") as demo:
    gr.Markdown("## Prompt-based detection on drone images\nPick a VisDrone image (densest first) or upload your own, then ask about it.")
    picked = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1):
            choice = gr.Dropdown(CHOICES, label="VisDrone image", value=None)
            image = gr.Image(type="numpy", label="Image (or upload your own)", sources=["upload"], height=320)
            text = gr.Textbox(label="Ask", placeholder="how many cars are on the left?", lines=1)
            gr.Examples(EXAMPLES, inputs=text)
            mode = gr.Radio(list(MODES.values()) + ["Compare both"], value=MODES["fast"], label="Detector")
            with gr.Accordion("Settings", open=False):
                use_llm = gr.Checkbox(True, label="Use Qwen parser (off = rule-based parser)")
                thr = gr.Slider(-0.05, 0.9, value=-0.05, step=0.05,
                                label="Confidence threshold (left end = calibrated default per detector)")
            go = gr.Button("Run", variant="primary")
        with gr.Column(scale=2):
            out1 = gr.Image(label="Result", height=520)
            out2 = gr.Image(label="Accurate: K1 + E2 ensemble", visible=False, height=520)
            ans = gr.Markdown()
            with gr.Accordion("Diagnostics: how the question was understood", open=False):
                diag = gr.JSON()
    choice.change(pick, choice, [image, picked])
    image.upload(on_upload, image, [picked, choice])
    go.click(handle, [image, picked, text, mode, use_llm, thr], [out1, out2, ans, diag])
    text.submit(handle, [image, picked, text, mode, use_llm, thr], [out1, out2, ans, diag])

demo.queue().launch(share=True, debug=True)